# Feature Engineering Gold Layer

In [3]:
# =============================================================================
# CREATE GOLD CATALOG & DATABASE
# =============================================================================
spark.sql("CREATE DATABASE IF NOT EXISTS credit_risk_assessment.gold")

DataFrame[]

# Joining datasets

In [4]:
loan       = spark.table("credit_risk_assessment.silver.loans")
payment    = spark.table("credit_risk_assessment.silver.payment_transactions")
financial  = spark.table("credit_risk_assessment.silver.customer_financials")
customers  = spark.table("credit_risk_assessment.silver.customers")

features = loan.join(payment, "loan_id") \
               .join(financial, "customer_id") \
               .join(customers, "customer_id")

# Create risk features

In [5]:
from pyspark.sql.functions import col, when, round

# Flag loans with high days past due
features = features.withColumn(
    "high_dpd_flag",
    when(col("days_past_due") > 60, 1).otherwise(0)
)

# Debt to equity ratio
features = features.withColumn(
    "debt_to_equity_ratio",
    round(col("total_debt") / col("equity"), 4)
)

# Loan to income ratio
features = features.withColumn(
    "loan_to_income_ratio",
    round(col("loan_amount") / col("annual_income"), 4)
)

# Credit utilisation flag (credit score below 550 = high risk)
features = features.withColumn(
    "low_credit_score_flag",
    when(col("credit_score") < 550, 1).otherwise(0)
)

# Revenue decline risk flag
features = features.withColumn(
    "revenue_decline_flag",
    when(col("revenue_decline") == 1, 1).otherwise(0)
)

# Save Gold table

In [6]:
features.write.format("delta").mode("overwrite") \
    .saveAsTable("credit_risk_assessment.gold.loan_features")

In [7]:
features.show(2)

+-----------+-------+-----------+---------+-------------+----------------+----------------+----------+--------------+-------------+--------------+----------+--------+--------+---------------+-------------+---+-------------+-----------------+------------+-------------+--------------------+--------------------+---------------------+--------------------+
|customer_id|loan_id|loan_amount|loan_type|interest_rate|loan_term_months|collateral_value|payment_id|payment_amount|days_past_due|payment_status|total_debt|  equity| revenue|revenue_decline|customer_name|age|annual_income|employment_status|credit_score|high_dpd_flag|debt_to_equity_ratio|loan_to_income_ratio|low_credit_score_flag|revenue_decline_flag|
+-----------+-------+-----------+---------+-------------+----------------+----------------+----------+--------------+-------------+--------------+----------+--------+--------+---------------+-------------+---+-------------+-----------------+------------+-------------+--------------------+---

# Query the table

In [16]:
# In AIDP make sure cell type is set to Python not SQL

rows = spark.sql("SELECT * FROM credit_risk_assessment.gold.loan_features WHERE loan_id = 44").collect()

if len(rows) == 0:
    print("No records found for loan_id = 44")
else:
    d = rows[0].asDict()
    print("\n" + "="*45)
    print(f"  LOAN DETAILS - loan_id = 44")
    print("="*45)
    for field, value in d.items():
        print(f"  {field:<30} : {value}")
    print("="*45)


  LOAN DETAILS - loan_id = 44
  customer_id                    : 371
  loan_id                        : 374
  loan_amount                    : 320617.0
  loan_type                      : personal
  interest_rate                  : 7.72
  loan_term_months               : 120
  collateral_value               : 190732.0
  payment_id                     : 4599
  payment_amount                 : 3587.0
  days_past_due                  : 90
  payment_status                 : on-time
  total_debt                     : 220733.0
  equity                         : 61749.0
  revenue                        : 221350.0
  revenue_decline                : 0
  customer_name                  : Customer_371
  age                            : 52
  annual_income                  : 34263.0
  employment_status              : self-employed
  credit_score                   : 669
  high_dpd_flag                  : 1
  debt_to_equity_ratio           : 3.5747
  loan_to_income_ratio           : 9.3575
  low_credi